In [1]:
# General imports
import numpy as np
import pandas as pd
np.random.seed(seed=1337)

# Local imports
from python_funcs.aggregation import(
    f_mean,
    f_quantile,
    f_wmean,
    f_wquantile,
    f_wsum,
)
from python_funcs.merges import(
    decorator_merge_prints,
    merge_with_amounts,
    merge_left_as_base,
)
from python_funcs.preparation import (
    remove_nas,
    remove_above_thr,
    remove_zero_and_below,
    remove_given_vals,
    keep_given_vals,
    limit_to_range,
)
from python_funcs.misc import(
    pct_change_fallback,
)

# Preparation

In [2]:
# Test data frame with edge cases for each function
df = pd.DataFrame({
    "category": ["A", "B", "C", "A", "B", "C", "A", "B"],
    "score": [10.5, -3.0, 0.0, 25.0, np.nan, 7.2, 100.0, 0.5],
    "date": pd.to_datetime([
        "2024-01-01", "2024-02-15", "2024-03-10", "2024-04-20",
        "2024-05-05", "2024-06-30", "2024-12-31", "2024-07-15",
    ]),
    "amount": [1000, 200, 500, 3000, 750, 1500, 8000, 400],
})
display(df)

,category,score,date,amount
0,A,10.5,2024-01-01,1000
1,B,-3.0,2024-02-15,200
2,C,0.0,2024-03-10,500
3,A,25.0,2024-04-20,3000
4,B,NaN,2024-05-05,750
5,C,7.2,2024-06-30,1500
6,A,100.0,2024-12-31,8000
7,B,0.5,2024-07-15,400


In [3]:
# Remove_nas: drop rows where "score" is NaN
display(remove_nas(df, "score", amount_col="amount"))

# Remove_above_thr: drop rows where "score" exceeds 20
display(remove_above_thr(df, "score", thr=20.0, amount_col="amount"))

# Remove_zero_and_below: drop rows where "score" <= 0
display(remove_zero_and_below(df, "score", amount_col="amount"))

# remove_given_vals: exclude specific categories
display(remove_given_vals(df, "category", values=["A", "C"], amount_col="amount"))

# keep_given_vals: keep only specific categories
display(keep_given_vals(df, "category", values=["A"], amount_col="amount"))

# limit_to_range: keep rows where "date" falls within Q1 2024
display(limit_to_range(
    df,
    "date",
    range_start=pd.Timestamp("2024-01-01"),
    range_end=pd.Timestamp("2024-03-31"),
    amount_col="amount",
))


remove_nas for variable score:
Lost 1 (12.5%) observations, 750 (4.9%) in amount.


,category,score,date,amount
0,A,10.5,2024-01-01,1000
1,B,-3.0,2024-02-15,200
2,C,0.0,2024-03-10,500
3,A,25.0,2024-04-20,3000
5,C,7.2,2024-06-30,1500
6,A,100.0,2024-12-31,8000
7,B,0.5,2024-07-15,400



remove_above_thr for variable score:
Threshold is 20.0000
Lost 3 (37.5%) observations, 11,750 (76.5%) in amount.


,category,score,date,amount
0,A,10.5,2024-01-01,1000
1,B,-3.0,2024-02-15,200
2,C,0.0,2024-03-10,500
5,C,7.2,2024-06-30,1500
7,B,0.5,2024-07-15,400



remove_zero_and_below for variable score:
Lost 3 (37.5%) observations, 1,450 (9.4%) in amount.


...

,category,score,date,amount
1,B,-3.0,2024-02-15,200
4,B,NaN,2024-05-05,750
7,B,0.5,2024-07-15,400



keep_given_vals for variable category:
Kept values are:
A
Lost 5 (62.5%) observations, 3,350 (21.8%) in amount.


,category,score,date,amount
0,A,10.5,2024-01-01,1000
3,A,25.0,2024-04-20,3000
6,A,100.0,2024-12-31,8000



limit_to_range for variable date:
Lost 5 (62.5%) observations, 13,650 (88.9%) in amount.


,category,score,date,amount
0,A,10.5,2024-01-01,1000
1,B,-3.0,2024-02-15,200
2,C,0.0,2024-03-10,500


# Merges

## merge_with_amounts

In [4]:
df_left = pd.DataFrame(
    data=np.random.normal(size=(4, 2)),
    index=[["a", "a", "b", "b"], [1, 2] * 2],
    columns=["l_1", "l_2"]
)
df_right = pd.DataFrame(
    data=np.random.normal(size=(6, 1)),
    index=[["a", "a", "c", "c", "d", "d"], [1, 2] * 3],
    columns=["r_1"]
)
display(df_left)
display(df_right)

print("Left-merge")
display(merge_with_amounts(
    df_left,
    df_right,
    left_index=True,
    right_index=True,
    how="left",
))

print("Right-merge")
display(merge_with_amounts(
    df_left,
    df_right,
    left_index=True,
    right_index=True,
    how="right",
))

print("Inner-merge")
display(merge_with_amounts(
    df_left,
    df_right,
    left_index=True,
    right_index=True,
    how="inner",
))

print("Outer-merge")
display(merge_with_amounts(
    df_left,
    df_right,
    left_index=True,
    right_index=True,
    how="outer",
))

l_1       l_2
a 1 -0.703187 -0.490282
  2 -0.321814 -1.755079
b 1  0.206664 -2.011265
  2 -0.557251  0.337217

r_1
a 1  1.548836
  2 -1.370737
c 1  1.425291
  2 -0.279464
d 1 -0.559628
  2  1.186383

Left-merge
Appearances in merged frame: both: 2, left_only: 2


l_1       l_2       r_1
a 1 -0.703187 -0.490282  1.548836
  2 -0.321814 -1.755079 -1.370737
b 1  0.206664 -2.011265       NaN
  2 -0.557251  0.337217       NaN

Right-merge
Appearances in merged frame: both: 2, right_only: 4


l_1       l_2       r_1
a 1 -0.703187 -0.490282  1.548836
  2 -0.321814 -1.755079 -1.370737
c 1       NaN       NaN  1.425291
  2       NaN       NaN -0.279464
d 1       NaN       NaN -0.559628
  2       NaN       NaN  1.186383

Inner-merge
Appearances in merged frame: both: 2


l_1       l_2       r_1
a 1 -0.703187 -0.490282  1.548836
  2 -0.321814 -1.755079 -1.370737

Outer-merge
Appearances in merged frame: both: 2, left_only: 2, right_only: 4


l_1       l_2       r_1
a 1 -0.703187 -0.490282  1.548836
  2 -0.321814 -1.755079 -1.370737
b 1  0.206664 -2.011265       NaN
  2 -0.557251  0.337217       NaN
c 1       NaN       NaN  1.425291
  2       NaN       NaN -0.279464
d 1       NaN       NaN -0.559628
  2       NaN       NaN  1.186383

## merge_left_as_base

Uses left frame keys are defintion of base population. Asserts that keys in the returend frame match those of left frame.

In [5]:
df_left = pd.DataFrame(
    data=np.random.normal(size=(4, 2)),
    index=[["a", "a", "b", "b"], [1, 2] * 2],
    columns=["l_1", "l_2"]
)
df_right = pd.DataFrame(
    data=np.random.normal(size=(6, 1)),
    index=[["a", "a", "c", "c", "d", "d"], [1, 2] * 3],
    columns=["r_1"]
)
display(merge_left_as_base(
    df_left,
    df_right,
    left_index=True,
    right_index=True,
))

Appearances in merged frame: both: 2, left_only: 2


l_1       l_2       r_1
a 1  1.698519 -1.691220 -1.314653
  2 -0.699523  0.582963 -0.379612
b 1  0.978223 -1.217372       NaN
  2 -1.329395 -0.001455       NaN

## Using decorator_merge_prints with custom merge function

In [6]:
@decorator_merge_prints
def _my_merge(left, right):
    return pd.merge(left, right, on="id", how="outer", indicator=True)

df_left  = pd.DataFrame({
    "id": ["a", "b", "c", "d", "f"],
    "val_l": [4, 7, 1, 4, 9],
})
df_right = pd.DataFrame({
    "id": ["a", "b", "c", "e", "f"],
    "val_r": [1, 2, 3, 4, 5],
})
display(_my_merge(df_left, df_right))

Appearances in merged frame: both: 4, left_only: 1, right_only: 1


,id,val_l,val_r
0,a,4.0,1.0
1,b,7.0,2.0
2,c,1.0,3.0
3,d,4.0,NaN
4,e,NaN,4.0
5,f,9.0,5.0


# Aggregation

In [7]:
# Data without NAs
df = pd.DataFrame({
    "name":["F", "B", "F", "B", "F", "F", "B", "F"],
    "group":["1", "1", "1", "1", "2", "2", "2", "2"],
    "weight": [2.0, 4, 1, 5, 3, 4, 2, 3],
    "score" : [6.0, 5, 9, 8, 7, 5, 7, 10],
})

# Data with NAs
df_na = df.copy()
df_na.loc[[2], "score"] = np.nan
df_na.loc[[5], "group"] = np.nan
df_na.loc[[7], "weight"] = np.nan

display(df)
display(df_na)

,name,group,weight,score
0,F,1,2.0,6.0
1,B,1,4.0,5.0
2,F,1,1.0,9.0
3,B,1,5.0,8.0
4,F,2,3.0,7.0
5,F,2,4.0,5.0
6,B,2,2.0,7.0
7,F,2,3.0,10.0


,name,group,weight,score
0,F,1,2.0,6.0
1,B,1,4.0,5.0
2,F,1,1.0,NaN
3,B,1,5.0,8.0
4,F,2,3.0,7.0
5,F,NaN,4.0,5.0
6,B,2,2.0,7.0
7,F,2,NaN,10.0


Sanity checks for the calculations below:

**No NAs**

 - Group (B, 1): rows 1, 3 — scores=[5, 8], weights=[4, 5]
   - mean / f_mean = (5+8)/2 = 6.5
   - median (linear) = 6.5
   - f_quantile(0.5): sorted [5, 8], cum [0.5, 1.0], boundary → avg(5, 8) = 6.5
   - f_wmean = (5·4 + 8·5)/9 = 6.6667
   - f_wquantile(0.5): sorted [5(4), 8(5)], cum [4/9, 1.0], q=0.5 in 8-bucket → 8
   - f_wsum: 60
- Group (B, 2): row 6 — single obs 7 with weight 2
   - f_wsum 14
   - all other = 7
- Group (F, 1): rows 0, 2 — scores=[6, 9], weights=[2, 1]
  - mean / f_mean = 7.5
  - median (linear) = 7.5
  - f_quantile(0.5): sorted [6, 9], cum [0.5, 1.0], boundary → 7.5
  - f_wmean = (6·2 + 9·1)/3 = 7.0
  - f_wquantile(0.5): sorted [6(2), 9(1)], cum [2/3, 1.0], q=0.5 in 6-bucket → 6
  - f_wsum: 21
- Group (F, 2): rows 4, 5, 7 — scores=[7, 5, 10], weights=[3, 4, 3]
  - mean / f_mean = 22/3 = 7.3333
  - median (linear) = sorted[1] = 7
  - f_quantile(0.5): sorted [5, 7, 10], cum [1/3, 2/3, 1.0], q=0.5 in 7-bucket → 7
  - f_wmean = (7·3 + 5·4 + 10·3)/10 = 7.1
  - f_wquantile(0.5): sorted [5(4), 7(3), 10(3)], cum [0.4, 0.7, 1.0], q=0.5 in 7-bucket → 7
  - f_wsum: 71

**NAs not ignored**

 - Group (B, 1): Same as in *No NAs*.
 - Group (B, 2): Same as in *No NAs*.
 - Group (F, 1): rows 0, 2 — scores=[6, NaN], weights=[2, 1]
   - mean (pandas) = 6
   - f_mean: score has NaN → NaN
   - f_wmean: score has NaN → NaN
   - median (pandas) = 6
   - f_quantile(0.5): score has NaN → NaN
   - f_wquantile(0.5): score has NaN → NaN
   - f_wsum: score has NaN → NaN
 - Group (F, 2): rows 4, 7 — scores=[7, 10], weights=[3, NaN]
   - mean (pandas) = 8.5
   - f_mean: no score-NaN → 8.5
   - f_wmean: weight has NaN → NaN
   - median (pandas) = 8.5
   - f_quantile(0.5): no score-NaN, sorted [7, 10], cum [0.5, 1.0], boundary → 8.5
   - f_wquantile(0.5): weight has NaN → NaN
   - f_wsum: weight has NaN → NaN
 - Group (F, NaN): row 5 — single obs 5 with weight 4, no NAs
   - f_wsum 20
   - all other = 5

**NAs ignored**

 - Group (B, 1): Same as in *No NAs*.
 - Group (B, 2): Same as in *No NAs*.
 - Group (F, 1): rows 0, 2 — scores=[6, NaN], weights=[2, 1]
   - mean (pandas) = 6, median (pandas) = 6
   - f_mean(ignore_na=True): drop row 2 → 6
   - f_quantile(0.5, ignore_na=True): drop row 2 → [6] → 6
   - f_wmean(ignore_na=True): drop row 2 → scores=[6], weights=[2] → 6
   - f_wquantile(0.5, ignore_na=True): single obs after drop → 6
   - f_wsum: 12
 - Group (F, 2): rows 4, 7 — scores=[7, 10], weights=[3, NaN]
   - mean (pandas) = 8.5, median (pandas) = 8.5
   - f_mean (strict): no score-NaN → 8.5
   - f_quantile(0.5, ignore_na=True): no score-NaN, same as Case 2 → 8.5
   - f_wmean(ignore_na=True): drop row 7 (NaN weight) → scores=[7], weights=[3] → 7
   - f_wquantile(0.5, ignore_na=True): single obs after drop → 7
   - f_wsum: 21
 - Group (F, NaN): row 5 — single obs 5 with weight 4, no NAs
   - f_wsum 20
   - all other = 5


In [8]:
# Without any NAs
display(
    df
    .groupby(["name", "group"], dropna=False)
    .agg({
        "weight": [
            "count"
        ],
        "score": [
            "count",
            "mean",
            f_mean(),
            f_wmean(weights=df["weight"]),
            "median",
            f_quantile(q=0.5),
            f_wquantile(q=0.5, weights=df["weight"]),
            f_wsum(weights=df["weight"]),
        ],
    })
)

# After setting NAs
display(
    df_na
    .groupby(["name", "group"], dropna=False)
    .agg({
        "weight": [
            "count"
        ],
        "score": [
            "count",
            "mean",
            f_mean(),
            f_wmean(weights=df_na["weight"]),
            "median",
            f_quantile(q=0.5),
            f_wquantile(q=0.5, weights=df_na["weight"]),
            f_wsum(weights=df_na["weight"]),
        ],
    })
)

# After setting NAs and ignoring them
display(
    df_na
    .groupby(["name", "group"], dropna=False)
    .agg({
        "weight": [
            "count"
        ],
        "score": [
            "count",
            "mean",
            f_mean(ignore_na=True),
            f_wmean(weights=df_na["weight"], ignore_na=True),
            "median",
            f_quantile(q=0.5, ignore_na=True),
            f_wquantile(q=0.5, weights=df_na["weight"], ignore_na=True),
            f_wsum(weights=df_na["weight"], ignore_na=True),
        ],
    })
)


weight score                                                    \
            count count      mean      mean     wmean median quantile_0.5   
name group                                                                  
B    1          2     2  6.500000  6.500000  6.666667    6.5          6.5   
     2          1     1  7.000000  7.000000  7.000000    7.0          7.0   
F    1          2     2  7.500000  7.500000  7.000000    7.5          7.5   
     2          3     3  7.333333  7.333333  7.100000    7.0          7.0   

                                
           wquantile_0.5  wsum  
name group                      
B    1               8.0  60.0  
     2               7.0  14.0  
F    1               6.0  21.0  
     2               7.0  71.0

weight score                                                        \
            count count mean mean     wmean median quantile_0.5 wquantile_0.5   
name group                                                                      
B    1          2     2  6.5  6.5  6.666667    6.5          6.5           8.0   
     2          1     1  7.0  7.0  7.000000    7.0          7.0           7.0   
F    1          2     1  6.0  NaN       NaN    6.0          NaN           NaN   
     2          1     2  8.5  8.5       NaN    8.5          8.5           NaN   
     NaN        1     1  5.0  5.0  5.000000    5.0          5.0           5.0   

                  
            wsum  
name group        
B    1      60.0  
     2      14.0  
F    1       NaN  
     2       NaN  
     NaN    20.0

weight score                                                        \
            count count mean mean     wmean median quantile_0.5 wquantile_0.5   
name group                                                                      
B    1          2     2  6.5  6.5  6.666667    6.5          6.5           8.0   
     2          1     1  7.0  7.0  7.000000    7.0          7.0           7.0   
F    1          2     1  6.0  6.0  6.000000    6.0          6.0           6.0   
     2          1     2  8.5  8.5  7.000000    8.5          8.5           7.0   
     NaN        1     1  5.0  5.0  5.000000    5.0          5.0           5.0   

                  
            wsum  
name group        
B    1      60.0  
     2      14.0  
F    1      12.0  
     2      21.0  
     NaN    20.0

# Misc

## pct_change_fallback

In [9]:
df = pd.DataFrame({
    "id":["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"],
    "val1":[1.0, 2.0, 4.0, 1.0e-6, 0.5e-9, 3.0, -4.0, -4.0, -1.0e-6, -0.5e-9],
    "val2":[1.0, 5.0, 2.0, 1.1, 1.1, -4.0, 2.0, 5.0, 1.1, 1.1],
})
df["change"] = pct_change_fallback(df["val1"], df["val2"])
display(df.style.format({
    "val1": "{:.3f}",
    "val2": "{:.3f}",
    "change": "{:.0%}",
}))

,id,val1,val2,change
0,1,1.000,1.000,0%
1,2,2.000,5.000,150%
2,3,4.000,2.000,-50%
3,4,0.000,1.100,109999900%
4,5,0.000,1.100,200%
5,6,3.000,-4.000,-233%
6,7,-4.000,2.000,150%
7,8,-4.000,5.000,225%
8,9,-0.000,1.100,110000100%
9,10,-0.000,1.100,200%
